<a href="https://colab.research.google.com/github/copyrightFreeGenAI/copyrightFreeImagesGenAI/blob/main/3.%20Fine-Tuning/Generate_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Set Paths
annValInstanceFiles = "/content/gdrive/MyDrive/annotations2014/instances_train2014.json" # Path to benchmark instance json
annValCaptionFiles = "/content/gdrive/MyDrive/annotations2014/captions_train2014.json" # Path to benchmark caption json
source_directory = "/content/gdrive/MyDrive/DATA/COCO/Validation/Images" # Path to directory where benchmark MS COCO images are saved

DREAM_WORKSPACE = '/content/gdrive/MyDrive/Fast-Dreambooth/Sessions' # Path to DreamBooth sessions
DREAM_target_directory = "/content/gdrive/MyDrive/Dream/Dream" # Path to store DreamBooth Images
LORA_WORKSPACE = '/content/gdrive/MyDrive/Loras' # Path to LoRA sessions
LORA_target_directory = "/content/gdrive/MyDrive/LoRA/LoRA" # Path to store LoRA Images
TI_WORKSPACE = '/content/gdrive/MyDrive/Textual-Inversion' # Path to Textual Inversion sessions
TI_target_directory = "/content/gdrive/MyDrive/Textual-Inversion/Textual-Inversion" # Path to store TI Images

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
from pycocotools.coco import COCO
import os
from diffusers import StableDiffusionPipeline, DiffusionPipeline

In [ ]:
coco=COCO(annValInstanceFiles)
coco_caps=COCO(annValCaptionFiles)
cats = coco.loadCats(coco.getCatIds())

# **Mitsua**

# Generate DreamBooth

In [ ]:
for cat in cats:
  print(cat['name'])
  category = cat['name'].replace(" ", "_")
  img_target = os.path.join(DREAM_target_directory, cat['name'])
  checkpoint_path = os.path.join(DREAM_WORKSPACE, category, category+".ckpt")
  if not os.path.exists(checkpoint_path):
    continue
  catIds = coco.getCatIds(catNms=[cat['name']])
  imgIds = coco.getImgIds(catIds=catIds)
  filenames = os.listdir(os.path.join(source_directory, cat['name']))
  names_before_dot = [int(os.path.splitext(file)[0]) for file in filenames]
  os.makedirs(img_target, exist_ok=True)
  if len(os.listdir(img_target)) != 100:
    pipeline = StableDiffusionPipeline.from_single_file(checkpoint_path, config="Mitsua/mitsua-diffusion-one", safety_checker=None).to("cuda")
    for image_id in names_before_dot:
      img_path = os.path.join(img_target, str(image_id) + ".jpg")
      if os.path.exists(img_path):
            continue
      img = coco.loadImgs(imgIds[image_id])[0]
      annIds = coco_caps.getAnnIds(imgIds=img['id'])
      anns = coco_caps.loadAnns(annIds)
      captions_list = [item['caption'].lower() for item in anns]
      prompt = "qwer. " + ' '.join(captions_list)
      prompt = prompt.replace(cat['name'], "qwer")
      image = pipeline(prompt).images[0]
      image.save(img_path)
    del pipeline

# Generate LoRA

In [ ]:
for cat in cats:
  print(cat['name'])
  category = cat['name'].replace(" ", "_")
  img_target = os.path.join(LORA_target_directory, cat['name'])
  checkpoint_path = os.path.join(LORA_WORKSPACE, category, 'output', category+"-20.safetensors")
  if not os.path.exists(checkpoint_path):
    continue
  catIds = coco.getCatIds(catNms=[cat['name']])
  imgIds = coco.getImgIds(catIds=catIds)
  filenames = os.listdir(os.path.join(source_directory, cat['name']))
  names_before_dot = [int(os.path.splitext(file)[0]) for file in filenames]
  os.makedirs(img_target, exist_ok=True)
  if len(os.listdir(img_target)) != 100:
    pipeline = DiffusionPipeline.from_pretrained("Mitsua/mitsua-diffusion-one", safety_checker=None).to("cuda")
    pipeline.load_lora_weights(checkpoint_path)
    for image_id in names_before_dot:
      img_path = os.path.join(img_target, str(image_id) + ".jpg")
      if os.path.exists(img_path):
            continue
      img = coco.loadImgs(imgIds[image_id])[0]
      annIds = coco_caps.getAnnIds(imgIds=img['id'])
      anns = coco_caps.loadAnns(annIds)
      captions_list = [item['caption'].lower() for item in anns]
      prompt = "qwer. " + ' '.join(captions_list)
      prompt = prompt.replace(cat['name'], "qwer")
      image = pipeline(prompt).images[0]
      image.save(img_path)
    del pipeline

# Generate Textual Inversion

In [ ]:
for category in os.listdir(source_directory):
  print(category)
  img_target = os.path.join(TI_target_directory, category)
  os.makedirs(img_target, exist_ok=True)
  if len(os.listdir(img_target)) == 100:
    continue
  dir_path = os.path.join(TI_WORKSPACE, category, 'learned_embeds.bin')
  if not os.path.exists(dir_path):
    continue
  pipeline = DiffusionPipeline.from_pretrained("Mitsua/mitsua-diffusion-one", safety_checker = None).to("cuda")
  pipeline.load_textual_inversion(dir_path)
  catIds = coco.getCatIds(catNms=[category])
  imgIds = coco.getImgIds(catIds=catIds)
  filenames = os.listdir(os.path.join(source_directory, category))
  names_before_dot = [int(os.path.splitext(file)[0]) for file in filenames]
  for image_id in names_before_dot:
      img_path = os.path.join(img_target, str(image_id) + ".jpg")
      if os.path.exists(img_path):
        continue
      img = coco.loadImgs(imgIds[image_id])[0]
      annIds = coco_caps.getAnnIds(imgIds=img['id'])
      anns = coco_caps.loadAnns(annIds)
      captions_list = [item['caption'].lower() for item in anns]
      prompt = category + ' ' + ' '.join(captions_list)
      prompt = prompt.replace(category, f"<{category}>")
      image = pipeline(prompt).images[0]
      image.save(img_path)
  del pipeline

# **Stable Diffusion 2.1**

## DreamBooth

In [ ]:
from pycocotools.coco import COCO
import os
from diffusers import StableDiffusionPipeline

In [ ]:
annFile="/content/gdrive/MyDrive/annotations2014/instances_train2014.json"
coco=COCO(annFile)
annFile = "/content/gdrive/MyDrive/annotations2014/captions_train2014.json"
coco_caps=COCO(annFile)
cats = coco.loadCats(coco.getCatIds())

In [ ]:
origin_directory = "/content/gdrive/MyDrive/Fast-Dreambooth/Sessions"
source_directory = "/content/gdrive/MyDrive/DATA/COCO/Validation/Images"
target_directory = "/content/gdrive/MyDrive/SD2.1/Dream/Dream"

In [ ]:
for cat in cats:
  print(cat['name'])
  category = cat['name'].replace(" ", "_")
  img_taget = os.path.join(target_directory, cat['name'])
  checkpoint_path = os.path.join(origin_directory, category, category+".ckpt")
  if not os.path.exists(checkpoint_path):
    continue
  catIds = coco.getCatIds(catNms=[cat['name']])
  imgIds = coco.getImgIds(catIds=catIds)
  filenames = os.listdir(os.path.join(source_directory, cat['name']))
  names_before_dot = [int(os.path.splitext(file)[0]) for file in filenames]
  os.makedirs(img_taget, exist_ok=True)
  if len(os.listdir(img_taget)) != 100:
    pipeline = StableDiffusionPipeline.from_single_file(checkpoint_path, config="Manojb/stable-diffusion-2-1-base", safety_checker=None).to("cuda")
    for image_id in names_before_dot:
      img_path = os.path.join(img_taget, str(image_id) + ".jpg")
      if os.path.exists(img_path):
            continue
      img = coco.loadImgs(imgIds[image_id])[0]
      annIds = coco_caps.getAnnIds(imgIds=img['id'])
      anns = coco_caps.loadAnns(annIds)
      captions_list = [item['caption'].lower() for item in anns]
      prompt = "qwer. " + ' '.join(captions_list)
      prompt = prompt.replace(cat['name'], "qwer")
      image = pipeline(prompt).images[0]
      image.save(img_path)
    del pipeline

## LoRA

In [ ]:
from pycocotools.coco import COCO
import os
from diffusers import StableDiffusionPipeline
import torch
from safetensors.torch import load_file

In [ ]:
annFile="/content/gdrive/MyDrive/annotations2014/instances_train2014.json"
coco=COCO(annFile)
annFile = "/content/gdrive/MyDrive/annotations2014/captions_train2014.json"
coco_caps=COCO(annFile)
cats = coco.loadCats(coco.getCatIds())

In [ ]:
origin_directory = "/content/gdrive/MyDrive/Loras"
source_directory = "/content/gdrive/MyDrive/DATA/COCO/Validation/Images"
target_directory = "/content/gdrive/MyDrive/SD2.1/LoRA/images"

In [ ]:
for cat in cats:
  print(cat['name'])
  category = cat['name'].replace(" ", "_")
  img_taget = os.path.join(target_directory, cat['name'])
  checkpoint_path = os.path.join(origin_directory, category, 'output', category+"-20.safetensors")
  if not os.path.exists(checkpoint_path):
    continue
  catIds = coco.getCatIds(catNms=[cat['name']])
  imgIds = coco.getImgIds(catIds=catIds)
  filenames = os.listdir(os.path.join(source_directory, cat['name']))
  names_before_dot = [int(os.path.splitext(file)[0]) for file in filenames]
  os.makedirs(img_taget, exist_ok=True)
  if len(os.listdir(img_taget)) != 100:
    pipeline = StableDiffusionPipeline.from_pretrained(
        "Manojb/stable-diffusion-2-1-base",
        torch_dtype=torch.float16,
        safety_checker=None,
        use_safetensors=True,
    ).to("cuda")
    sd = load_file(checkpoint_path)
    pipeline.load_lora_weights(sd, adapter_name="lora")
    pipeline.set_adapters(["lora"], adapter_weights=[0.2])
    for image_id in names_before_dot:
      img_path = os.path.join(img_taget, str(image_id) + ".jpg")
      if os.path.exists(img_path):
            continue
      img = coco.loadImgs(imgIds[image_id])[0]
      annIds = coco_caps.getAnnIds(imgIds=img['id'])
      anns = coco_caps.loadAnns(annIds)
      captions_list = [item['caption'].lower() for item in anns]
      prompt = "qwer. " + ' '.join(captions_list)
      prompt = prompt.replace(cat['name'], "qwer")
      image = pipeline(prompt).images[0]
      image.save(img_path)
    del pipeline

## Textual Inversion

In [ ]:
from pycocotools.coco import COCO
import os
import pandas as pd
from diffusers import DiffusionPipeline

In [ ]:
annFile="/content/gdrive/MyDrive/annotations2014/instances_train2014.json"
coco=COCO(annFile)
annFile = "/content/gdrive/MyDrive/annotations2014/captions_train2014.json"
coco_caps=COCO(annFile)
cats = coco.loadCats(coco.getCatIds())

In [ ]:
target_directory = "/content/gdrive/MyDrive/SD2.1/TI/TI"
source_directory = "/content/gdrive/MyDrive/DATA/COCO/Validation/Images"
image_directory = "/content/gdrive/MyDrive/SD2.1/TI/Images"

In [ ]:
for category in os.listdir(source_directory):
  print(category)
  img_target = os.path.join(image_directory, category)
  os.makedirs(img_target, exist_ok=True)
  if len(os.listdir(img_target)) == 100:
    continue
  dir_path = os.path.join(target_directory, category, 'learned_embeds.bin')
  if not os.path.exists(dir_path):
    continue
  pipeline = DiffusionPipeline.from_pretrained("Manojb/stable-diffusion-2-1-base", safety_checker = None).to("cuda")
  pipeline.load_textual_inversion(dir_path)
  catIds = coco.getCatIds(catNms=[category])
  imgIds = coco.getImgIds(catIds=catIds)
  filenames = os.listdir(os.path.join(source_directory, category))
  names_before_dot = [int(os.path.splitext(file)[0]) for file in filenames]
  for image_id in names_before_dot:
      img_path = os.path.join(img_target, str(image_id) + ".jpg")
      if os.path.exists(img_path):
        continue
      img = coco.loadImgs(imgIds[image_id])[0]
      annIds = coco_caps.getAnnIds(imgIds=img['id'])
      anns = coco_caps.loadAnns(annIds)
      captions_list = [item['caption'].lower() for item in anns]
      prompt = category + ' ' + ' '.join(captions_list)
      prompt = prompt.replace(category, f"<{category}>")
      image = pipeline(prompt).images[0]
      image.save(img_path)
  del pipeline

# **Juggernaut XL**

## DreamBooth

In [ ]:
from pycocotools.coco import COCO
import os
from diffusers import DiffusionPipeline
import torch

In [ ]:
annFile="/content/gdrive/MyDrive/annotations2014/instances_train2014.json"
coco=COCO(annFile)
annFile = "/content/gdrive/MyDrive/annotations2014/captions_train2014.json"
coco_caps=COCO(annFile)
cats = coco.loadCats(coco.getCatIds())

In [ ]:
origin_directory = "/content/gdrive/MyDrive/JuggXL/Dream/models"
source_directory = "/content/gdrive/MyDrive/DATA/COCO/Validation/Images"
target_directory = "/content/gdrive/MyDrive/JuggXL/Dream/Dream"

In [ ]:
pipeline = DiffusionPipeline.from_pretrained(
        "RunDiffusion/Juggernaut-XL-v9",
        torch_dtype=torch.float16,
        variant="fp16",
        safety_checker=None,
        use_safetensors=True
    ).to("cuda")

In [ ]:
for cat in cats:
  print(cat['name'])
  img_taget = os.path.join(target_directory, cat['name'])
  checkpoint_path = os.path.join(origin_directory, cat['name'], "pytorch_lora_weights.safetensors")
  if not os.path.exists(checkpoint_path):
    # training sessions store multi-word categories with underscores
    checkpoint_path = os.path.join(origin_directory, cat['name'].replace(" ", "_"), "pytorch_lora_weights.safetensors")
  if not os.path.exists(checkpoint_path):
    continue
  catIds = coco.getCatIds(catNms=[cat['name']])
  imgIds = coco.getImgIds(catIds=catIds)
  filenames = os.listdir(os.path.join(source_directory, cat['name']))
  names_before_dot = [int(os.path.splitext(file)[0]) for file in filenames]
  os.makedirs(img_taget, exist_ok=True)
  if len(os.listdir(img_taget)) != 100:
    pipeline.unload_lora_weights()
    pipeline.load_lora_weights(checkpoint_path)
    for image_id in names_before_dot:
      img_path = os.path.join(img_taget, str(image_id) + ".jpg")
      if os.path.exists(img_path):
            continue
      img = coco.loadImgs(imgIds[image_id])[0]
      annIds = coco_caps.getAnnIds(imgIds=img['id'])
      anns = coco_caps.loadAnns(annIds)
      captions_list = [item['caption'].lower() for item in anns]
      prompt = "qwer. " + ' '.join(captions_list)
      prompt = prompt.replace(cat['name'], "qwer")
      image = pipeline(prompt).images[0]
      image.save(img_path)

## LoRA

In [ ]:
from pycocotools.coco import COCO
import os
from diffusers import StableDiffusionXLPipeline
import torch
from safetensors.torch import load_file

In [ ]:
annFile="/content/gdrive/MyDrive/annotations2014/instances_train2014.json"
coco=COCO(annFile)
annFile = "/content/gdrive/MyDrive/annotations2014/captions_train2014.json"
coco_caps=COCO(annFile)
cats = coco.loadCats(coco.getCatIds())

In [ ]:
origin_directory = "/content/gdrive/MyDrive/Loras"
source_directory = "/content/gdrive/MyDrive/DATA/COCO/Validation/Images"
target_directory = "/content/gdrive/MyDrive/JuggXL/LoRA/images"

In [ ]:
for cat in cats:
  print(cat['name'])
  category = cat['name'].replace(" ", "_")
  img_taget = os.path.join(target_directory, cat['name'])
  checkpoint_path = os.path.join(origin_directory, category, 'output', category+"-20.safetensors")
  if not os.path.exists(checkpoint_path):
    continue
  catIds = coco.getCatIds(catNms=[cat['name']])
  imgIds = coco.getImgIds(catIds=catIds)
  filenames = os.listdir(os.path.join(source_directory, cat['name']))
  names_before_dot = [int(os.path.splitext(file)[0]) for file in filenames]
  os.makedirs(img_taget, exist_ok=True)
  if len(os.listdir(img_taget)) != 100:
    pipeline = StableDiffusionXLPipeline.from_pretrained(
        "RunDiffusion/Juggernaut-XL-v9",
        torch_dtype=torch.float16,
        safety_checker=None,
        use_safetensors=True,
        variant="fp16"
    ).to("cuda")
    sd = load_file(checkpoint_path)
    pipeline.load_lora_weights(sd, adapter_name="lora")
    pipeline.set_adapters(["lora"])
    for image_id in names_before_dot:
      img_path = os.path.join(img_taget, str(image_id) + ".jpg")
      if os.path.exists(img_path):
            continue
      img = coco.loadImgs(imgIds[image_id])[0]
      annIds = coco_caps.getAnnIds(imgIds=img['id'])
      anns = coco_caps.loadAnns(annIds)
      captions_list = [item['caption'].lower() for item in anns]
      prompt = "qwer. " + ' '.join(captions_list)
      prompt = prompt.replace(cat['name'], "qwer")
      image = pipeline(prompt).images[0]
      image.save(img_path)
    del pipeline

## Textual Inversion

In [ ]:
from pycocotools.coco import COCO
import os
import pandas as pd
import torch
from diffusers import DiffusionPipeline

In [ ]:
annFile="/content/gdrive/MyDrive/annotations2014/instances_train2014.json"
coco=COCO(annFile)
annFile = "/content/gdrive/MyDrive/annotations2014/captions_train2014.json"
coco_caps=COCO(annFile)
cats = coco.loadCats(coco.getCatIds())

In [ ]:
target_directory = "/content/gdrive/MyDrive/JuggXL/TI/TI"
source_directory = "/content/gdrive/MyDrive/DATA/COCO/Validation/Images"
image_directory = "/content/gdrive/MyDrive/JuggXL/TI/Images"

In [ ]:
for category in os.listdir(source_directory):
  print(category)
  # the trainer registers an underscored token and may store the folder that way too
  token = f"<{category.replace(' ', '_')}>"
  embeds_dir = os.path.join(target_directory, category)
  if not os.path.isdir(embeds_dir):
    embeds_dir = os.path.join(target_directory, category.replace(" ", "_"))
  dir_path = os.path.join(embeds_dir, 'learned_embeds-steps-2000.safetensors')
  dir_path_2 = os.path.join(embeds_dir, 'learned_embeds_2-steps-2000.safetensors')
  if not os.path.exists(dir_path):
    # the final checkpoint may be written without the step suffix
    dir_path = os.path.join(embeds_dir, 'learned_embeds.safetensors')
    dir_path_2 = os.path.join(embeds_dir, 'learned_embeds_2.safetensors')
  if not (os.path.exists(dir_path) and os.path.exists(dir_path_2)):
    continue
  img_target = os.path.join(image_directory, category)
  os.makedirs(img_target, exist_ok=True)
  if len(os.listdir(img_target)) == 100:
    continue
  pipeline = DiffusionPipeline.from_pretrained(
      "RunDiffusion/Juggernaut-XL-v9",
      torch_dtype=torch.float16,
      variant="fp16",
      use_safetensors=True,
      safety_checker=None
  ).to("cuda")
  pipeline.load_textual_inversion(
      dir_path,
      token=token,
      tokenizer=pipeline.tokenizer,
      text_encoder=pipeline.text_encoder,
  )

  pipeline.load_textual_inversion(
      dir_path_2,
      token=token,
      tokenizer=pipeline.tokenizer_2,
      text_encoder=pipeline.text_encoder_2,
  )
  catIds = coco.getCatIds(catNms=[category])
  imgIds = coco.getImgIds(catIds=catIds)
  filenames = os.listdir(os.path.join(source_directory, category))
  names_before_dot = [int(os.path.splitext(file)[0]) for file in filenames]
  for image_id in names_before_dot:
      img_path = os.path.join(img_target, str(image_id) + ".jpg")
      if os.path.exists(img_path):
        continue
      img = coco.loadImgs(imgIds[image_id])[0]
      annIds = coco_caps.getAnnIds(imgIds=img['id'])
      anns = coco_caps.loadAnns(annIds)
      captions_list = [item['caption'].lower() for item in anns]
      prompt = category + ' ' + ' '.join(captions_list)
      prompt = prompt.replace(category, token)
      image = pipeline(prompt).images[0]
      image.save(img_path)
  del pipeline
  torch.cuda.empty_cache()